# 🧠 Neuro-Symbolic Test-Time Training (NS-TTT)

**Dual-speed transformer architecture** that adapts at inference time using symbolic constraint violations as gradients.

- **System 1 (Base):** Frozen Gemma-3-1B-IT
- **System 2 (Symbolic):** Rule-based constraint engine
- **Bridge:** Gumbel-Softmax concept projection
- **Adaptation:** LoRA fast weights updated via `L_TTT = L_self_sup + λ·L_sym`

---
⚡ **Runtime:** T4 GPU (free Colab tier)

## 1. Setup

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/aiworldmodel.git
%cd aiworldmodel

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 2. Run Tests (No Model Download Required)

In [ ]:
# Run the full test suite — uses mock models, no download needed
!python -m pytest tests/ -v --tb=short

## 3. Initialize the Pipeline

This downloads Gemma-3-1B-IT (~2 GB) on first run.

In [ ]:
from inference.pipeline import NeuroSymbolicTTTPipeline

# Load from config
pipeline = NeuroSymbolicTTTPipeline.from_config(
    config_path="config/ttt_config.yaml",
    device="cuda",
)
print(pipeline)

## 4. Generate with TTT Enabled

The model will:
1. Reset LoRA adapters
2. Run N dry-run forward passes
3. Project hidden states → symbolic concepts via Gumbel-Softmax
4. Compute `L_TTT` and update fast weights
5. Generate final output with adapted weights

In [ ]:
# TTT-enabled generation
result = pipeline.generate(
    prompt="The patient presents with persistent cough, fever of 39.2°C, and bilateral lung infiltrates on chest X-ray. Diagnosis:",
    ttt_enabled=True,
)

print("=" * 60)
print("🤖 Response:")
print(result.output_text)
print("=" * 60)

In [ ]:
# View detailed TTT metrics
from utils.metrics import TTTMetrics

metrics = TTTMetrics.from_ttt_result(result)
print(metrics.summary())

## 5. Compare: TTT vs Standard Inference

In [ ]:
prompt = "A 45-year-old male with type 2 diabetes and hypertension presents with sudden onset chest pain radiating to the left arm. ECG shows ST elevation in leads V1-V4. Management:"

# Standard inference (no TTT)
result_standard = pipeline.generate(prompt, ttt_enabled=False)
print("📌 Standard Inference:")
print(result_standard.output_text)
print(f"   Time: {result_standard.total_time_ms:.0f}ms")
print()

# TTT-enabled inference
result_ttt = pipeline.generate(prompt, ttt_enabled=True)
print("🧠 TTT-Enabled Inference:")
print(result_ttt.output_text)
print(f"   Time: {result_ttt.total_time_ms:.0f}ms")
print(f"   TTT Steps: {len(result_ttt.step_metrics)}")
print(f"   Final Adapter Norm: {result_ttt.final_adapter_norm:.4f}")

## 6. Inspect TTT Dynamics

In [ ]:
import matplotlib.pyplot as plt

metrics = TTTMetrics.from_ttt_result(result_ttt)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("TTT Dynamics", fontsize=14, fontweight="bold")

steps = range(len(metrics.loss_per_step))

# Loss curves
axes[0, 0].plot(steps, metrics.loss_per_step, 'b-o', label='Total')
axes[0, 0].plot(steps, metrics.self_sup_loss_per_step, 'g--', label='Self-Sup')
axes[0, 0].plot(steps, metrics.symbolic_loss_per_step, 'r--', label='Symbolic')
axes[0, 0].set_title('Loss per Step')
axes[0, 0].legend()
axes[0, 0].set_xlabel('TTT Step')

# Gradient norms
axes[0, 1].plot(steps, metrics.grad_norm_per_step, 'm-o')
axes[0, 1].set_title('Gradient Norm')
axes[0, 1].set_xlabel('TTT Step')

# Adapter norm
axes[1, 0].plot(steps, metrics.adapter_norm_per_step, 'c-o')
axes[1, 0].set_title('Adapter Norm (‖BA‖_F)')
axes[1, 0].set_xlabel('TTT Step')

# Temperature
axes[1, 1].plot(steps, metrics.temperature_per_step, 'orange', marker='o')
axes[1, 1].set_title('Gumbel-Softmax Temperature τ')
axes[1, 1].set_xlabel('TTT Step')

plt.tight_layout()
plt.show()

## 7. Custom Symbolic Rules

In [ ]:
from symbolic.world_model import SymbolicWorldModel, MutualExclusionRule, TemporalOrderRule

# Create a custom world model
custom_wm = SymbolicWorldModel()

# Add domain-specific rules
custom_wm.add_rule(MutualExclusionRule(
    exclusion_pairs=[(10, 25), (30, 45)],
    weight=1.5,
))

custom_wm.add_rule(TemporalOrderRule(
    ordered_concepts=[5, 10, 15, 20],
    weight=1.0,
))

print(custom_wm)

## 8. Save Adapter State (Episodic Memory)

In [ ]:
# Save the adapted weights for potential offline consolidation
pipeline.save_session("saved_sessions/medical_demo.pt")
print("Session saved!")

# Reset to base model
pipeline.reset_session()
print("Session reset — base model restored.")

## 9. Cleanup

In [ ]:
# Free GPU memory
pipeline.unload()
torch.cuda.empty_cache()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")